# Análisis del comportamiento de una aplicación bajo diferentes escenarios de despliegue

**Curso:** Principios y prácticas de Desarrollo de Software Orientado a Objetos — Prof. Gabriel Rodrigo Pedraza Ferreira

**Integrantes:**
- Thomas Perez
- David Suarez
- Jesus Santodomingo

**Repositorio:** _(enlace)_ · **Notebook en Colab:** _(enlace)_

## 1. Introducción y objetivos

Este trabajo mide cómo cambian el **throughput**, el **tiempo de respuesta** y la **tasa de error** de una API REST
cuando se despliega con Docker Compose y con Kubernetes (K3s) usando 1, 2 y 3 nodos y 1, 2 y 3 réplicas.

**Objetivos:**
- Medir el costo (overhead) de la orquestación con Kubernetes frente a Docker Compose sobre el mismo hardware.
- Evaluar el escalado horizontal: ¿crece el throughput al agregar réplicas y nodos?
- Identificar los cuellos de botella (CPU de la API, base de datos, punto de entrada de red).

## 2. Descripción de la aplicación

**Dominio:** tienda online. API REST en **FastAPI** (Python 3.12, 1 worker de Uvicorn por contenedor) con **PostgreSQL 16**.

| Tabla | Registros | Índices |
|---|---|---|
| productos | 100.000 | PK, `categoria` |
| clientes | 20.000 | PK |
| pedidos | 200.000 | PK, `cliente_id` |
| items_pedido | ~600.000 | PK, `pedido_id`, `producto_id` |

Los datos se generan con Faker y semilla fija (`seed.py`), por lo que son idénticos en cada escenario.
La BD se resetea antes de cada escenario.

| Método | Ruta | Tipo de carga | Peso |
|---|---|---|---|
| GET | `/productos/{id}` | lectura por clave primaria | 50 % |
| GET | `/productos?categoria=X&page=N` | lectura paginada con filtro | 25 % |
| GET | `/productos/buscar?q=texto` | `ILIKE` en nombre y descripción + conteo (recorre toda la tabla) | 10 % |
| POST | `/pedidos` | escritura transaccional (pedido + 2 items) | 10 % |
| GET | `/reportes/ventas-por-categoria` | `JOIN` + `GROUP BY` sobre ~600K items | 5 % |

```
clientes 1───* pedidos 1───* items_pedido *───1 productos
```

## 3. Infraestructura

4 VMs en **Google Cloud** `n2d-standard-2` (2 vCPU AMD EPYC, 8 GB, no burstable; equivalente al `Standard_D2as_v5` de Azure), Ubuntu 24.04, región `us-central1`, misma red privada `10.0.1.0/24` (nodo1 `.10`, nodo2 `.11`, nodo3 `.12`, vm-jmeter `.20`).

_Se intentó primero con Azure for Students, pero la cuota de la cuenta (6 vCPUs, 0 para la familia DASv5) no alcanzaba y Azure rechazó el aumento._

```
 ┌──────────────┐    HTTP     ┌──────────────────────────────────────────────┐
 │  vm-jmeter   │ ──────────► │ nodo1 (Fase 1: Docker Compose)               │
 │  JMeter CLI  │             │ nodo1 (K3s server + PostgreSQL               │
 └──────────────┘             │        + Prometheus/Grafana + pods de la API) │
                              │ nodo2 (K3s agent, 2 nodos)                    │
                              │ nodo3 (K3s agent, 3 nodos)                    │
                              └──────────────────────────────────────────────┘
```

## 4. Configuración de despliegue por escenario

| Recurso | Docker Compose | Kubernetes |
|---|---|---|
| API | `cpus: 1.0`, `mem_limit: 512m` | `limits: cpu 1, 512Mi` · `requests: cpu 250m` |
| PostgreSQL | `cpus: 1.0`, `mem_limit: 2g` | `limits: cpu 1, 2Gi` · `requests: cpu 500m`, fijo en nodo1 |
| Pool de conexiones | 10 + 5 overflow | 10 + 5 overflow por réplica |
| Entrada | puerto 8000 de nodo1 | NodePort 30080 de nodo1 |

Los `requests` de CPU se bajaron respecto a los `limits` para que las 3 réplicas quepan en el escenario de 1 nodo;
los **límites** (lo que realmente puede consumir cada contenedor) son iguales a los de Docker.

_(Agregar capturas de `kubectl get pods -o wide` de cada escenario.)_

## 5. Configuración de la monitorización

- **Docker:** Prometheus + Grafana + cAdvisor + node-exporter (`docker-compose.monitoring.yml`).
- **Kubernetes:** `kube-prometheus-stack` (Helm) con Alertmanager desactivado, Prometheus y Grafana fijos en nodo1,
  node-exporter en todos los nodos y un `ServiceMonitor` para `/metrics` de la API.
- Las métricas de cada corrida se exportaron a CSV con `scripts/exportar_prometheus.py` usando los tiempos de `results/runs.csv`.

## 6. Metodología JMeter

- JMeter 5.6.3 en modo CLI desde `vm-jmeter` (VM distinta a la aplicación, misma red).
- Mezcla de peticiones con Throughput Controllers (50/25/10/10/5 %), datos de `ids.csv`, aserción de código 200/201.
- Niveles de carga: **10, 50, 100 y 200 usuarios**; **3 repeticiones** por nivel; 180 s por corrida con 30 s de ramp-up.
- Calentamiento de 60 s descartado al inicio de cada escenario y 30 s de enfriamiento entre corridas.
- Se descartan los primeros 30 s (ramp-up) de cada corrida al calcular las métricas.
- Reglas de rigor: BD reseteada antes de cada escenario, pods `Ready`, CPU de `vm-jmeter` vigilada (< 80 %).

## 7. Resultados

### 7.1 Carga de datos

In [ ]:
import os, glob, re
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

REPO_URL = "https://github.com/Tapr16/Proyecto-Final-Principios.git"

try:
    import google.colab  # noqa: F401
    EN_COLAB = True
except ImportError:
    EN_COLAB = False

if EN_COLAB and not Path("proyecto-final").exists():
    os.system(f"git clone -q {REPO_URL} proyecto-final")

RESULTS = Path(os.environ.get("PF_RESULTS", "proyecto-final/results" if EN_COLAB else "../results"))
print("Resultados en:", RESULTS.resolve())

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 110
RAMP_MS = 30_000
ORDEN = ["docker"] + [f"k8s-{n}n-{r}r" for n in (1, 2, 3) for r in (1, 2, 3)]

In [ ]:
filas, filas_ep = [], []
for f in sorted(glob.glob(str(RESULTS / "*" / "u*_r*.jtl"))):
    esc = Path(f).parent.name
    u, r = map(int, re.findall(r"u(\d+)_r(\d+)\.jtl$", f)[0])
    df = pd.read_csv(f, low_memory=False)
    df["success"] = df["success"].astype(str).str.lower() == "true"
    df = df[df.timeStamp >= df.timeStamp.min() + RAMP_MS]      # descartar ramp-up
    dur = (df.timeStamp.max() - df.timeStamp.min()) / 1000
    m = re.match(r"k8s-(\d)n-(\d)r", esc)
    base = {
        "escenario": esc,
        "plataforma": "Docker" if esc == "docker" else "K8s",
        "nodos": int(m.group(1)) if m else 1,
        "replicas": int(m.group(2)) if m else 1,
        "usuarios": u, "rep": r,
    }
    filas.append({**base,
        "peticiones": len(df),
        "rt_medio_ms": df.elapsed.mean(),
        "rt_p95_ms": df.elapsed.quantile(0.95),
        "throughput_rps": df.success.sum() / dur,
        "error_pct": 100 * (1 - df.success.mean()),
    })
    for label, g in df.groupby("label"):
        filas_ep.append({**base, "endpoint": label,
                         "rt_medio_ms": g.elapsed.mean(),
                         "rt_p95_ms": g.elapsed.quantile(0.95),
                         "error_pct": 100 * (1 - g.success.mean())})

res = pd.DataFrame(filas)
res_ep = pd.DataFrame(filas_ep)
assert len(res), f"No se encontraron .jtl en {RESULTS}"
print(f"{len(res)} corridas cargadas")
res.groupby("escenario").size().reindex([e for e in ORDEN if e in set(res.escenario)]).to_frame("corridas")

### 7.2 Tabla resumen (media ± desviación estándar de las 3 repeticiones)

In [ ]:
claves = ["escenario", "plataforma", "nodos", "replicas", "usuarios"]
agg = res.groupby(claves).agg(
    tps=("throughput_rps", "mean"), tps_std=("throughput_rps", "std"),
    rt=("rt_medio_ms", "mean"), rt_std=("rt_medio_ms", "std"),
    p95=("rt_p95_ms", "mean"), p95_std=("rt_p95_ms", "std"),
    err=("error_pct", "mean"), err_std=("error_pct", "std"),
).reset_index().fillna({"tps_std": 0, "rt_std": 0, "p95_std": 0, "err_std": 0})
agg["escenario"] = pd.Categorical(agg.escenario, [e for e in ORDEN if e in set(agg.escenario)], ordered=True)
agg = agg.sort_values(["escenario", "usuarios"])
CARGA_MAX = agg.usuarios.max()

tabla = agg.copy()
for c in ["tps", "rt", "p95", "err"]:
    tabla[c] = tabla[c].round(1).astype(str) + " ± " + tabla[c + "_std"].round(1).astype(str)
tabla[["escenario", "usuarios", "tps", "rt", "p95", "err"]].rename(columns={
    "tps": "throughput (req/s)", "rt": "t. medio (ms)", "p95": "p95 (ms)", "err": "error (%)"})

### 7.3 Throughput y tiempo de respuesta vs. usuarios (una figura por número de nodos)

In [ ]:
def por_nodos(col, std, ylabel, titulo):
    fig, axes = plt.subplots(1, 3, figsize=(16, 4.2), sharey=True)
    docker = agg[agg.escenario == "docker"]
    for ax, n in zip(axes, (1, 2, 3)):
        sub = agg[(agg.plataforma == "K8s") & (agg.nodos == n)]
        for r, g in sub.groupby("replicas"):
            ax.errorbar(g.usuarios, g[col], yerr=g[std], marker="o", capsize=3, label=f"{r} réplica(s)")
        if len(docker):
            ax.errorbar(docker.usuarios, docker[col], yerr=docker[std], marker="s", ls="--",
                        color="gray", capsize=3, label="Docker (ref.)")
        ax.set_title(f"K8s — {n} nodo(s)")
        ax.set_xlabel("usuarios concurrentes")
        ax.legend(fontsize=8)
    axes[0].set_ylabel(ylabel)
    fig.suptitle(titulo)
    fig.tight_layout()
    plt.show()

por_nodos("tps", "tps_std", "throughput (req/s)", "Gráfica 1 — Throughput vs. usuarios")
por_nodos("rt", "rt_std", "tiempo medio de respuesta (ms)", "Gráfica 2 — Tiempo de respuesta vs. usuarios")

### 7.4 Heatmap nodos × réplicas (throughput con la carga máxima)

In [ ]:
k8s_max = agg[(agg.plataforma == "K8s") & (agg.usuarios == CARGA_MAX)]
hm = k8s_max.pivot_table(index="nodos", columns="replicas", values="tps")
plt.figure(figsize=(5.5, 4))
sns.heatmap(hm, annot=True, fmt=".0f", cmap="viridis", cbar_kws={"label": "req/s"})
plt.title(f"Gráfica 3 — Throughput con {CARGA_MAX} usuarios")
plt.show()

### 7.5 Speedup al aumentar réplicas

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4.2))
for n, g in k8s_max.groupby("nodos"):
    base = g[g.replicas == 1]
    if base.empty:
        continue
    b, b_std = base.tps.iloc[0], base.tps_std.iloc[0]
    g = g.sort_values("replicas")
    s = g.tps / b
    # propagación del error de un cociente: σs = s·√((σa/a)² + (σb/b)²)
    s_std = s * np.sqrt((g.tps_std / g.tps) ** 2 + (b_std / b) ** 2)
    ax.errorbar(g.replicas, s, yerr=s_std, marker="o", capsize=3, label=f"{n} nodo(s)")
ax.plot([1, 2, 3], [1, 2, 3], "k--", label="ideal lineal")
ax.set_xticks([1, 2, 3]); ax.set_xlabel("réplicas"); ax.set_ylabel("speedup")
ax.set_title(f"Gráfica 4 — Speedup = TPS(N)/TPS(1) con {CARGA_MAX} usuarios")
ax.legend(); plt.show()

### 7.6 Docker vs. K8s 1 nodo 1 réplica (costo de la orquestación)

In [ ]:
comp = agg[agg.escenario.isin(["docker", "k8s-1n-1r"])]
if comp.escenario.nunique() == 2:
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    for ax, (col, std, lab) in zip(axes, [("tps", "tps_std", "throughput (req/s)"),
                                          ("rt", "rt_std", "tiempo medio (ms)")]):
        piv = comp.pivot_table(index="usuarios", columns="escenario", values=col, observed=True)
        err = comp.pivot_table(index="usuarios", columns="escenario", values=std, observed=True)
        piv.plot.bar(yerr=err, ax=ax, capsize=3, rot=0)
        ax.set_ylabel(lab)
    fig.suptitle("Gráfica 5 — Docker vs. K8s (1 nodo, 1 réplica)")
    fig.tight_layout(); plt.show()

    t = comp.pivot_table(index="usuarios", columns="escenario", values="tps", observed=True)
    print("Overhead de K8s en throughput (%):")
    print((100 * (t["docker"] - t["k8s-1n-1r"]) / t["docker"]).round(1).to_string())
else:
    print("Faltan datos de docker o de k8s-1n-1r")

### 7.7 Mismas réplicas, distinto número de nodos (escalado horizontal)

In [ ]:
tres = agg[agg.escenario.isin(["k8s-1n-3r", "k8s-2n-3r", "k8s-3n-3r"])]
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for esc, g in tres.groupby("escenario", observed=True):
    axes[0].errorbar(g.usuarios, g.tps, yerr=g.tps_std, marker="o", capsize=3, label=esc)
    axes[1].errorbar(g.usuarios, g.rt, yerr=g.rt_std, marker="o", capsize=3, label=esc)
axes[0].set_ylabel("throughput (req/s)"); axes[1].set_ylabel("tiempo medio (ms)")
for ax in axes:
    ax.set_xlabel("usuarios concurrentes"); ax.legend()
fig.suptitle("Gráfica 6 — 3 réplicas en 1, 2 y 3 nodos")
fig.tight_layout(); plt.show()

### 7.8 Tasa de error por escenario y carga

In [ ]:
err = agg.pivot_table(index="escenario", columns="usuarios", values="err", observed=True)
plt.figure(figsize=(7, 5))
sns.heatmap(err, annot=True, fmt=".2f", cmap="Reds", cbar_kws={"label": "% error"})
plt.title("Gráfica 7 — Tasa de error (%)")
plt.show()

### 7.9 ¿Qué endpoint se degrada más?

In [ ]:
ep = res_ep.groupby(["escenario", "endpoint", "usuarios"]).rt_medio_ms.mean().reset_index()
carga_min = ep.usuarios.min()
deg = (ep[ep.usuarios == CARGA_MAX].set_index(["escenario", "endpoint"]).rt_medio_ms
       / ep[ep.usuarios == carga_min].set_index(["escenario", "endpoint"]).rt_medio_ms).unstack()
deg = deg.reindex([e for e in ORDEN if e in deg.index])
plt.figure(figsize=(9, 5))
sns.heatmap(deg, annot=True, fmt=".1f", cmap="magma_r",
            cbar_kws={"label": f"t. medio({CARGA_MAX}) / t. medio({carga_min})"})
plt.title("Degradación del tiempo de respuesta por endpoint")
plt.xticks(rotation=25, ha="right"); plt.show()

ep.pivot_table(index=["escenario", "usuarios"], columns="endpoint", values="rt_medio_ms").round(1)

## 8. Métricas internas (Prometheus)

Archivos `results/prometheus/<escenario>_u<U>_r<R>_<metrica>.csv` generados por `scripts/exportar_prometheus.py`.
Se promedia cada serie durante la corrida (sin los primeros 30 s).

In [ ]:
prom_files = glob.glob(str(RESULTS / "prometheus" / "*.csv"))
if not prom_files:
    print("No hay CSV de Prometheus todavía")
else:
    partes = []
    for f in prom_files:
        m = re.match(r"(.+)_u(\d+)_r(\d+)_(\w+)\.csv$", Path(f).name)
        df = pd.read_csv(f)
        if df.empty:
            continue
        df = df[df.ts >= df.ts.min() + RAMP_MS / 1000]
        g = df.groupby("serie").valor.mean().reset_index()
        g[["escenario", "usuarios", "rep", "metrica"]] = m.group(1), int(m.group(2)), int(m.group(3)), m.group(4)
        partes.append(g)
    prom = pd.concat(partes, ignore_index=True)

    def tipo(serie):
        s = str(serie)
        if "postgres" in s or "-db-" in s:
            return "PostgreSQL"
        if s.startswith("api") or "-api-" in s:
            return "API"
        return "otro"
    prom["tipo"] = prom.serie.map(tipo)
    prom_max = prom[prom.usuarios == CARGA_MAX]
    orden_p = [e for e in ORDEN if e in set(prom_max.escenario)]

    def barras(ax, por_rep, col, escala=1):
        # media ± desviación estándar entre las repeticiones
        g = por_rep.groupby(["escenario", col]).valor
        g.mean().mul(escala).unstack().reindex(orden_p).plot.bar(
            ax=ax, rot=45, capsize=3,
            yerr=g.std().fillna(0).mul(escala).unstack().reindex(orden_p))

    def por_tipo(metrica):   # suma de las réplicas de cada tipo, por repetición
        d = prom_max[(prom_max.metrica == metrica) & (prom_max.tipo != "otro")]
        return d.groupby(["escenario", "rep", "tipo"]).valor.sum().reset_index()

    fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))
    barras(axes[0], por_tipo("cpu_pod"), "tipo")
    axes[0].set_ylabel("núcleos de CPU"); axes[0].set_title("CPU total: API vs. PostgreSQL")

    barras(axes[1], prom_max[prom_max.metrica == "cpu_nodo"], "serie", escala=100)
    axes[1].set_ylabel("% CPU"); axes[1].set_title("CPU por nodo")

    barras(axes[2], por_tipo("throttling_pod"), "tipo")
    axes[2].set_ylabel("periodos limitados / s"); axes[2].set_title("Throttling de CPU")
    fig.suptitle(f"Gráfica 8 — Métricas internas con {CARGA_MAX} usuarios")
    fig.tight_layout(); plt.show()

## 9. Análisis comparativo

_Responder con base en las gráficas anteriores:_

1. ¿El throughput crece al aumentar réplicas? ¿De forma lineal o se aplana? (Gráficas 1, 3 y 4)
2. ¿En qué carga se satura cada configuración (el throughput deja de subir y el tiempo de respuesta se dispara)? (Gráficas 1 y 2)
3. ¿3 réplicas en 1 nodo rinden igual que 3 réplicas en 3 nodos? ¿Por qué? (Gráfica 6 y CPU por nodo)
4. ¿La base de datos es el cuello de botella? (Gráfica 8: CPU de PostgreSQL vs. CPU de la API)
5. ¿Cuánto overhead añade Kubernetes frente a Docker Compose? (Gráfica 5)
6. ¿Agregar nodos ayuda si la BD sigue siendo una sola instancia?
7. ¿Qué endpoint se degrada más? (sección 7.9)
8. Limitante: todo el tráfico entra por el NodePort de nodo1.

## 10. Conclusiones

_(completar)_

## 11. Recomendaciones

_(validar cada una con los datos)_

- Escalar la BD: réplicas de lectura, PgBouncer o caché con Redis.
- Horizontal Pod Autoscaler según la CPU observada.
- Ajustar `requests` y `limits`.
- Ingress o balanceador externo en vez del NodePort de un solo nodo.
- Índice trigram (`pg_trgm`) para la búsqueda `ILIKE`.